# Eval Original PowerPaint Model
Chi danh gia model goc (JunhaoZhuang/PowerPaint_v2) tren COCO val

In [1]:
from google.colab import drive
from pathlib import Path
import os, sys, json, yaml, shutil, subprocess

drive.mount('/content/gdrive')

DRIVE_WORKSPACE = Path('/content/gdrive/MyDrive/Eval_Original')
REPO_DIR = DRIVE_WORKSPACE / 'PowerPaint'
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

print('Python =', sys.version)
print('REPO_DIR =', REPO_DIR)
print('WORKSPACE =', DRIVE_WORKSPACE)

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
Python = 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
REPO_DIR = /content/gdrive/MyDrive/Eval_Original/PowerPaint
WORKSPACE = /content/gdrive/MyDrive/Eval_Original


In [ ]:
# Config
base_model = str(REPO_DIR / 'checkpoints' / 'ppt-v2' / 'realisticVisionV60B1_v51VAE')
powerpaint_ckpt = str(REPO_DIR / 'checkpoints' / 'ppt-v2' / 'PowerPaint_Brushnet')
results_root = DRIVE_WORKSPACE / 'results' / 'original'
pp_steps = 45
pp_guidance_removal = 12
pp_guidance_inpaint = 7.5
pp_run_text_guided = True
pp_text_guided_use_dataset_caption = True
pp_text_guided_prompt = 'a red bench'  # Chi dung khi pp_text_guided_use_dataset_caption = False
pp_negative_prompt_text_guided = ''
pp_prompt_removal = ''
pp_negative_prompt_removal = ''
pp_mask_dilate_px = 2
pp_blur_radius = 1
pp_feather_edge_px = 6
pp_infer_image_size, pp_eval_image_size = 512, 512 # Dat None de giu kich thuoc anh goc (se lam tron ve boi so cua 8) 
pp_skip_existing = True  # Neu output da ton tai thi bo qua mau do

if pp_infer_image_size is not None:
    assert pp_infer_image_size % 8 == 0, 'pp_infer_image_size phai chia het cho 8'
if pp_eval_image_size is not None:
    assert pp_eval_image_size > 0, 'pp_eval_image_size phai lon hon 0'

VAL_ZIP_PATH = "/content/gdrive/MyDrive/COCO_Raw_Data/coco_clean_val.zip"
BASE_DATA_DIR = "/content/coco_clean_splits"
SPLIT = "val"
DATASET_DIR = os.path.join(BASE_DATA_DIR, SPLIT)

os.makedirs(BASE_DATA_DIR, exist_ok=True)

if not os.path.exists(os.path.join(DATASET_DIR, "metadata.json")):
    !cp "{VAL_ZIP_PATH}" /content/val_data.zip
    !unzip -q /content/val_data.zip -d {BASE_DATA_DIR}
    !rm /content/val_data.zip

In [3]:
# Cai cac goi can thiet cho Colab theo bo version on dinh hon
import sys
import subprocess
import importlib.metadata as importlib_metadata
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Hay chuyen Colab sang GPU truoc khi chay notebook nay.')

py312_plus = sys.version_info >= (3, 12)
transformers_version = '4.39.3' if py312_plus else '4.28.0'
tokenizers_package = ['tokenizers==0.15.2'] if py312_plus else []

required_versions = {
    'diffusers': '0.27.0',
    'transformers': transformers_version,
    'huggingface_hub': '0.20.2',
    'accelerate': '0.31.0',
    'controlnet-aux': '0.0.3',
    'pillow': '10.3.0',
    'torchmetrics': '0.11.4',
    'torch-fidelity': '0.3.0',
}

def installed_version(dist_name):
    try:
        return importlib_metadata.version(dist_name)
    except importlib_metadata.PackageNotFoundError:
        return None

core_packages = [
    'diffusers==0.27.0',
    f'transformers=={transformers_version}',
    'huggingface_hub==0.20.2',
    'accelerate==0.31.0',
    'controlnet-aux==0.0.3',
    'safetensors>=0.4.3',
    'pillow==10.3.0',
    *tokenizers_package,
]

extra_packages = [
    'mmengine', 'omegaconf', 'packaging', 'imageio', 'opencv-python-headless',
    'albumentations', 'pycocotools', 'torchmetrics==0.11.4', 'torch-fidelity==0.3.0',
    'tqdm', 'pandas', 'pyyaml', 'ftfy', 'scipy', 'sentencepiece', 'einops', 'timm',
]

print('Python =', sys.version)
print('Checking package versions...')
mismatches = {name: (installed_version(name), want) for name, want in required_versions.items() if installed_version(name) != want}

# peft moi san tren Colab co the xung dot voi accelerate 0.31.0 cua PowerPaint.
peft_version = installed_version('peft')
if peft_version is not None:
    print(f'Removing incompatible peft=={peft_version} ...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'peft'])

if mismatches:
    print('Installing/upgrading mismatched packages only:')
    for name, (have, want) in mismatches.items():
        print(f'  - {name}: {have} -> {want}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', '--no-cache-dir', *core_packages])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', '--no-cache-dir', *extra_packages])
else:
    print('Required package versions already installed.')

import os

# Kiểm tra xem pillow có nằm trong danh sách các gói vừa bị sai phiên bản và phải cài lại không
if mismatches and 'pillow' in mismatches:
    old_version = mismatches['pillow'][0]
    new_version = mismatches['pillow'][1]
    print(f"Pillow vừa được cập nhật (từ {old_version} lên {new_version}). Đang tự động khởi động lại session...")

    # Ngắt tiến trình hiện tại để Colab tự động restart
    os.kill(os.getpid(), 9)
else:
    print("Pillow đã đúng phiên bản (10.3.0). Không cần khởi động lại session, bạn có thể chạy tiếp.")

Python = 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Checking package versions...
Required package versions already installed.
Pillow đã đúng phiên bản (10.3.0). Không cần khởi động lại session, bạn có thể chạy tiếp.


In [4]:
import numpy as np
import torch
import pandas as pd
from PIL import Image, ImageChops, ImageFilter
from tqdm.auto import tqdm
from safetensors.torch import load_model
from transformers import CLIPTextModel
from powerpaint.models.BrushNet_CA import BrushNetModel
from powerpaint.models.unet_2d_condition import UNet2DConditionModel
from powerpaint.pipelines.pipeline_PowerPaint_Brushnet_CA import StableDiffusionPowerPaintBrushNetPipeline
from powerpaint.utils.utils import TokenizerWrapper, add_tokens, EmbeddingLayerWithFixes

print('Torch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
print('Transformers:', __import__('transformers').__version__)
print('Diffusers:', __import__('diffusers').__version__)
print('Pillow:', __import__('PIL').__version__)

Torch: 2.11.0+cu128 CUDA: True
Transformers: 4.39.3
Diffusers: 0.27.0
Pillow: 10.3.0


In [5]:
# Load original model
print("Loading original PowerPaint model...")

unet = UNet2DConditionModel.from_pretrained('runwayml/stable-diffusion-v1-5', subfolder='unet', torch_dtype=torch.float16)
text_encoder_brushnet = CLIPTextModel.from_pretrained('runwayml/stable-diffusion-v1-5', subfolder='text_encoder', torch_dtype=torch.float16)
brushnet = BrushNetModel.from_unet(unet).to('cuda', dtype=torch.float16)

tokenizer = TokenizerWrapper(from_pretrained=base_model, subfolder='tokenizer')
raw_tokenizer = tokenizer.wrapped
add_tokens(
    tokenizer=tokenizer,
    text_encoder=text_encoder_brushnet,
    placeholder_tokens=["P_ctxt", "P_obj"],
    initialize_tokens=["a", "a"],
    num_vectors_per_token=10,
)

load_model(brushnet, os.path.join(powerpaint_ckpt, 'diffusion_pytorch_model.safetensors'))
text_encoder_brushnet.load_state_dict(
    torch.load(os.path.join(powerpaint_ckpt, 'pytorch_model.bin')), strict=False
)

pipe = StableDiffusionPowerPaintBrushNetPipeline.from_pretrained(
    base_model,
    brushnet=brushnet,
    unet=UNet2DConditionModel.from_pretrained(base_model, subfolder='unet', torch_dtype=torch.float16),
    text_encoder_brushnet=text_encoder_brushnet,
    tokenizer=raw_tokenizer,
    safety_checker=None,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=False,
)

# promptU / negative_promptU di qua pipe.text_encoder, nen can dong bo external embeddings
main_embedding_layer = pipe.text_encoder.text_model.embeddings.token_embedding
if not isinstance(main_embedding_layer, EmbeddingLayerWithFixes):
    pipe.text_encoder.text_model.embeddings.token_embedding = EmbeddingLayerWithFixes(main_embedding_layer)
    main_embedding_layer = pipe.text_encoder.text_model.embeddings.token_embedding

copied_embeddings = []
for emb in text_encoder_brushnet.text_model.embeddings.token_embedding.external_embeddings:
    copied = {k: v for k, v in emb.items() if k not in {'embedding'}}
    copied['embedding'] = emb['embedding'].detach().clone()
    copied['trainable'] = False
    copied_embeddings.append(copied)

if copied_embeddings:
    main_embedding_layer.add_embeddings(copied_embeddings)

pipe.tokenizer = tokenizer
pipe = pipe.to('cuda')
pipe.set_progress_bar_config(disable=True)
print("Model loaded")


Loading original PowerPaint model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/492M [00:00<?, ?B/s]

06/29 15:42:11 - mmengine - INFO - Successfully add external embeddings: P_ctxt, P_obj.
06/29 15:42:11 - mmengine - INFO - Successfully add trainable external embeddings: P_ctxt, P_obj


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/clip/feature_extraction_clip.py:28: FutureWarning: The class CLIPFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use CLIPImageProcessor instead.
  warnings.warn(


06/29 15:45:30 - mmengine - INFO - Successfully add external embeddings: P_ctxt, P_obj.
Model loaded


In [15]:
# Helper functions
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def _list_images(folder):
    folder = Path(folder)
    return sorted([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS])

def _load_image(path, size=None):
    image = Image.open(path).convert("RGB")
    if size is not None:
        image = image.resize((size, size), Image.Resampling.BICUBIC)
    array = np.asarray(image, dtype=np.float32) / 255.0
    return torch.from_numpy(array).permute(2, 0, 1)

def _load_batch(paths, size, device):
    return torch.stack([_load_image(p, size=size) for p in paths], dim=0).to(device)

def _round_to_multiple_of_8(x):
    return max(8, (x // 8) * 8)

def _prepare_inference_inputs(image, mask, target_size=512, dilate_px=0):
    image = image.convert('RGB')
    mask = mask.convert('L').resize(image.size, Image.NEAREST)
    if target_size is None:
        target_w = _round_to_multiple_of_8(image.size[0])
        target_h = _round_to_multiple_of_8(image.size[1])
    else:
        target_size = _round_to_multiple_of_8(int(target_size))
        target_w = target_size
        target_h = target_size
    image = image.resize((target_w, target_h), Image.Resampling.LANCZOS)
    mask = mask.resize((target_w, target_h), Image.NEAREST)
    binary_mask = mask.point(lambda p: 255 if p > 127 else 0, mode='L')
    if dilate_px > 0:
        kernel_size = dilate_px * 2 + 1
        binary_mask = binary_mask.filter(ImageFilter.MaxFilter(size=kernel_size))
    masked_image = Image.composite(Image.new('RGB', image.size, (0,0,0)), image, binary_mask)
    return image, binary_mask, masked_image

# def _compose_inpainted_result(base_image, generated, mask, feather_edge_px=8):
#     generated = generated.convert("RGB").resize(base_image.size, Image.Resampling.LANCZOS)
#     base_image = base_image.convert("RGB")

#     mask_l = mask.convert("L")

#     # Feather toàn bộ mask, không tách hard_center + edge rối
#     alpha = mask_l.filter(ImageFilter.GaussianBlur(radius=feather_edge_px))
#     alpha_np = (np.asarray(alpha, dtype=np.float32) / 255.0)[..., None]

#     base_np = np.asarray(base_image, dtype=np.float32) / 255.0
#     gen_np = np.asarray(generated, dtype=np.float32) / 255.0

#     out = base_np * (1.0 - alpha_np) + gen_np * alpha_np
#     return Image.fromarray(np.uint8(np.clip(out * 255.0, 0, 255)))
def _compose_inpainted_result(base_image, generated, mask, blur_radius=1, feather_edge_px=6):
    # Debug mode: bo qua toan bo blend, tra ve anh generated sau khi resize
    # Neu anh nay van mo thi van de nam o model / resize truoc inference.
    # Neu anh nay net hon final cu, thi van de nam o buoc compose/blend.
    return generated.convert('RGB').resize(base_image.size, Image.Resampling.LANCZOS)

def _build_task_prompts(prompt, negative_prompt, task, version='ppt-v2'):
    pos_prefix = ''
    neg_prefix = ''
    if task in {'object-removal', 'image-outpainting'}:
        if version == 'ppt-v1':
            pos_prefix = 'empty scene blur ' + prompt
            neg_prefix = negative_prompt
        promptA = pos_prefix + ' P_ctxt'
        promptB = pos_prefix + ' P_ctxt'
        negative_promptA = neg_prefix + ' P_obj'
        negative_promptB = neg_prefix + ' P_obj'
    elif task == 'shape-guided':
        if version == 'ppt-v1':
            pos_prefix = prompt
            neg_prefix = negative_prompt + ', worst quality, low quality, normal quality, bad quality, blurry '
        promptA = pos_prefix + ' P_shape'
        promptB = pos_prefix + ' P_ctxt'
        negative_promptA = neg_prefix + 'P_shape'
        negative_promptB = neg_prefix + 'P_ctxt'
    else:
        if version == 'ppt-v1':
            pos_prefix = prompt
            neg_prefix = negative_prompt + ', worst quality, low quality, normal quality, bad quality, blurry '
        promptA = pos_prefix + ' P_obj'
        promptB = pos_prefix + ' P_obj'
        negative_promptA = neg_prefix + 'P_obj'
        negative_promptB = neg_prefix + 'P_obj'
    return promptA, promptB, negative_promptA, negative_promptB

def _save_debug_views(debug_dir, stem, original, mask, pipe_input, generated, result):
    from PIL import ImageDraw, ImageFont

    labels = ['original', 'mask', 'pipe_input', 'generated', 'final']
    panels = [
        original.convert('RGB'),
        mask.convert('RGB'),
        pipe_input.convert('RGB'),
        generated.convert('RGB').resize(original.size, Image.Resampling.LANCZOS),
        result.convert('RGB'),
    ]
    label_height = 32
    try:
        font = ImageFont.truetype('DejaVuSans.ttf', 18)
    except OSError:
        font = ImageFont.load_default()
    total_width = sum(panel.width for panel in panels)
    max_height = max(panel.height for panel in panels) + label_height
    canvas = Image.new('RGB', (total_width, max_height), (0, 0, 0))
    draw = ImageDraw.Draw(canvas)
    offset_x = 0
    for label, panel in zip(labels, panels):
        canvas.paste(panel, (offset_x, label_height))
        bbox = draw.textbbox((0, 0), label, font=font)
        text_w = bbox[2] - bbox[0]
        text_h = bbox[3] - bbox[1]
        text_x = offset_x + max(0, (panel.width - text_w) // 2)
        text_y = max(0, (label_height - text_h) // 2 - 1)
        draw.text((text_x, text_y), label, fill=(255, 255, 255), font=font)
        offset_x += panel.width
    out_path = Path(debug_dir) / f'{stem}_debug.jpg'
    canvas.save(out_path, quality=95)
    return out_path

def evaluate_pairs(pairs, prompts, batch_size=8, image_size=299, device=None, clip_model_name="openai/clip-vit-base-patch16"):
    from torchmetrics.image.fid import FrechetInceptionDistance
    from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity
    from torchmetrics.multimodal import CLIPScore

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    device = torch.device(device)

    real_paths = [x[0] for x in pairs]
    gen_paths = [x[1] for x in pairs]

    fid = FrechetInceptionDistance(feature=2048, normalize=True).to(device)
    for i in range(0, len(real_paths), batch_size):
        fid.update(_load_batch(real_paths[i:i+batch_size], image_size, device), real=True)
    for i in range(0, len(gen_paths), batch_size):
        fid.update(_load_batch(gen_paths[i:i+batch_size], image_size, device), real=False)
    fid_value = float(fid.compute().detach().cpu().item())

    lpips = LearnedPerceptualImagePatchSimilarity(net_type="alex", normalize=True).to(device)
    for i in range(0, len(pairs), batch_size):
        chunk = pairs[i:i+batch_size]
        rb = _load_batch([x[0] for x in chunk], image_size, device)
        gb = _load_batch([x[1] for x in chunk], image_size, device)
        lpips.update(gb, rb)
    lpips_value = float(lpips.compute().detach().cpu().item())

    clip = CLIPScore(model_name_or_path=clip_model_name).to(device)
    for i in range(0, len(gen_paths), batch_size):
        clip.update(_load_batch(gen_paths[i:i+batch_size], image_size, device), list(prompts[i:i+batch_size]))
    clip_value = float(clip.compute().detach().cpu().item())

    return {"fid": fid_value, "lpips": lpips_value, "clip_score": clip_value, "num_samples": len(pairs)}

def _collect_eval_pairs(items, generated_dir, task_name):
    generated_dir = Path(generated_dir)
    gen_map = {p.name: p for p in _list_images(generated_dir)}
    pairs, prompts = [], []
    for item in items:
        real_path = Path(item['image_path'])
        name = real_path.name
        if name not in gen_map:
            continue
        pairs.append((real_path, gen_map[name]))
        if task_name == 'text-guided' and pp_text_guided_use_dataset_caption:
            prompts.append(str(item.get('caption', '')).strip())
        elif task_name == 'text-guided':
            prompts.append(pp_text_guided_prompt)
        else:
            prompts.append('')
    return pairs, prompts

def _save_metrics(task_label, metrics, metrics_dir):
    metrics_dir = Path(metrics_dir)
    metrics_dir.mkdir(parents=True, exist_ok=True)
    row = pd.DataFrame([{**metrics, 'task': task_label}])
    csv_path = metrics_dir / f'{task_label}_metrics.csv'
    json_path = metrics_dir / f'{task_label}_metrics.json'
    row.to_csv(csv_path, index=False)
    with open(json_path, 'w') as f:
        json.dump({**metrics, 'task': task_label}, f, indent=2)
    return csv_path, json_path

In [7]:
# Unzip dataset
if not os.path.exists(os.path.join(DATASET_DIR, "metadata.json")):
    print("Extracting clean val dataset...")
    if os.path.exists(DATASET_DIR):
        shutil.rmtree(DATASET_DIR)
    os.makedirs(BASE_DATA_DIR, exist_ok=True)
    !cp "{VAL_ZIP_PATH}" /content/val_data.zip
    !unzip -q /content/val_data.zip -d {BASE_DATA_DIR}
    !rm /content/val_data.zip

with open(os.path.join(DATASET_DIR, "metadata.json"), "r") as f:
    full_metadata = json.load(f)

metadata_run = []
for item in full_metadata:
    img_filename = os.path.basename(item["image_path"])
    mask_filename = os.path.basename(item["mask_path"])
    metadata_run.append({
        "image_path": os.path.join(DATASET_DIR, "images", img_filename),
        "mask_path": os.path.join(DATASET_DIR, "masks", mask_filename),
        "caption": item.get("caption", "")
    })

METADATA_RUN_PATH = os.path.join(DATASET_DIR, "metadata_run.json")
with open(METADATA_RUN_PATH, "w") as f:
    json.dump(metadata_run, f, indent=2)

print(f"Samples: {len(metadata_run)}, metadata: {METADATA_RUN_PATH}")

Samples: 4952, metadata: /content/coco_clean_splits/val/metadata_run.json


In [16]:
# Inference Helper
with open(METADATA_RUN_PATH) as f:
    items = json.load(f)

def _run_powerpaint_task(items, task_name, prompt, negative_prompt, out_subdir, progress_desc, guidance_scale):
    out_dir = results_root / out_subdir
    debug_dir = results_root / f'{out_subdir}_debug'
    out_dir.mkdir(parents=True, exist_ok=True)
    debug_dir.mkdir(parents=True, exist_ok=True)

    skipped_missing_prompt = 0

    for item in tqdm(items, desc=progress_desc):
        output_name = Path(item['image_path']).name
        stem = Path(item['image_path']).stem
        final_path = out_dir / output_name
        debug_path = debug_dir / f'{stem}_debug.jpg'
        if pp_skip_existing and final_path.exists() and debug_path.exists():
            continue

        item_prompt = prompt
        if task_name == 'text-guided' and pp_text_guided_use_dataset_caption:
            item_prompt = str(item.get('caption', '')).strip()
        if task_name == 'text-guided' and not item_prompt:
            skipped_missing_prompt += 1
            continue

        promptA, promptB, negative_promptA, negative_promptB = _build_task_prompts(
            item_prompt, negative_prompt, task_name, version='ppt-v2'
        )
        promptU = item_prompt
        if task_name == 'object-removal':
            promptU = item_prompt + ' empty scene blur'

        img = Image.open(item['image_path']).convert('RGB')
        msk = Image.open(item['mask_path']).convert('L')
        image, mask, masked_img = _prepare_inference_inputs(
            img, msk,
            target_size=pp_infer_image_size,
            dilate_px=pp_mask_dilate_px,
        )
        out = pipe(
            promptA=promptA, promptB=promptB, promptU=promptU,
            negative_promptA=negative_promptA, negative_promptB=negative_promptB,
            negative_promptU=negative_prompt,
            tradoff=1.0, tradoff_nag=1.0, image=masked_img, mask=mask,
            num_inference_steps=pp_steps, guidance_scale=guidance_scale,
            brushnet_conditioning_scale=1.0,
            generator=torch.Generator(device='cuda').manual_seed(42),
            width=image.size[0], height=image.size[1],
        ).images[0]
        final = _compose_inpainted_result(
            image, out, mask,
            feather_edge_px=pp_feather_edge_px,
        )
        final.save(final_path)
        _save_debug_views(debug_dir, stem, image, mask, masked_img, out, final)

    print(f'Saved to {out_dir}')
    print(f'Saved debug views to {debug_dir}')
    if skipped_missing_prompt > 0:
        print(f'Skipped {skipped_missing_prompt} samples because caption/prompt was empty.')

print('Task helper ready.')

removal:   0%|          | 0/4952 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Inference: object_removal
_run_powerpaint_task(
    items,
    task_name='object-removal',
    prompt=pp_prompt_removal,
    negative_prompt=pp_negative_prompt_removal,
    out_subdir='object_removal',
    progress_desc='removal',
    guidance_scale=pp_guidance_removal,
)
object_pairs, object_prompts = _collect_eval_pairs(items, results_root / 'object_removal', 'object-removal')
print(f'Matched object_removal samples: {len(object_pairs)}')
object_result = evaluate_pairs(object_pairs, object_prompts, batch_size=8, image_size=pp_eval_image_size)
object_csv_path, object_json_path = _save_metrics('object_removal', object_result, results_root / 'metrics')
print(pd.DataFrame([{**object_result, 'task': 'object_removal'}]))
print(f'Saved metrics to {object_csv_path}')
print(f'Saved metrics to {object_json_path}')
print('Object removal completed.')

In [ ]:
# Inference: text_guided_object_synthesis
if pp_run_text_guided:
    if not pp_text_guided_use_dataset_caption and not pp_text_guided_prompt.strip():
        raise ValueError('Hay dat pp_text_guided_prompt truoc khi bat pp_run_text_guided=True')
    _run_powerpaint_task(
        items,
        task_name='text-guided',
        prompt=pp_text_guided_prompt,
        negative_prompt=pp_negative_prompt_text_guided,
        out_subdir='text_guided_object_synthesis',
        progress_desc='text-guided',
        guidance_scale=pp_guidance_inpaint,
    )
    text_pairs, text_prompts = _collect_eval_pairs(items, results_root / 'text_guided_object_synthesis', 'text-guided')
    print(f'Matched text_guided samples: {len(text_pairs)}')
    text_result = evaluate_pairs(text_pairs, text_prompts, batch_size=8, image_size=pp_eval_image_size)
    text_csv_path, text_json_path = _save_metrics('text_guided_object_synthesis', text_result, results_root / 'metrics')
    print(pd.DataFrame([{**text_result, 'task': 'text_guided_object_synthesis'}]))
    print(f'Saved metrics to {text_csv_path}')
    print(f'Saved metrics to {text_json_path}')
    print('Text-guided object synthesis completed.')
else:
    print('Skipped text-guided object synthesis because pp_run_text_guided=False.')

del pipe
torch.cuda.empty_cache()
print('Inference completed.')

In [ ]:
# Metrics summary
metrics_dir = results_root / 'metrics'
metric_files = sorted(metrics_dir.glob('*_metrics.csv')) if metrics_dir.exists() else []
if not metric_files:
    print('No metrics files found yet.')
else:
    summary_df = pd.concat([pd.read_csv(path) for path in metric_files], ignore_index=True)
    print(summary_df)
    summary_csv_path = metrics_dir / 'all_metrics_summary.csv'
    summary_df.to_csv(summary_csv_path, index=False)
    print(f'Saved metrics summary to {summary_csv_path}')